# Phase 1: Environment Setup & Raw Schema Inspection

### Objective
Initialize the processing environment and set up memory-efficient data scanning for the LendingClub dataset using Polars LazyFrames.

### Technical Rationale
The raw LendingClub CSV is multi-gigabyte containing over 2 million rows and 100+ feature columns. Standard Pandas `read_csv` attempts to load the entire payload into RAM at once, leading to excessive memory pressure or out-of-memory (OOM) crashes.
- We use `pl.scan_csv` to establish a query plan to explore the dataset

In [1]:
import polars as pl

# Polars display config
pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_cols(12)

# File path 
DATA_PATH = "/home/mike/Documents/work/DS/projects/credit-risk/data/raw/accepted_2007_to_2018Q4.csv"

# Step 1: Create the LazyFrame query graph
lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10000,
    ignore_errors=True
)

In [4]:
lazy_df

In [6]:
schema = lazy_df.collect_schema()
total_columns = len(schema.names())

print(f"Total Columns Loaded: {total_columns}")

Total Columns Loaded: 151


In [7]:
lazy_df.head(5).collect()

id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,…,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
i64,str,f64,f64,f64,str,…,str,str,str,f64,f64,f64
68407277,null,3600.0,3600.0,3600.0,""" 36 months""",…,null,null,null,null,null,null
68355089,null,24700.0,24700.0,24700.0,""" 36 months""",…,null,null,null,null,null,null
68341763,null,20000.0,20000.0,20000.0,""" 60 months""",…,null,null,null,null,null,null
66310712,null,35000.0,35000.0,35000.0,""" 60 months""",…,null,null,null,null,null,null
68476807,null,10400.0,10400.0,10400.0,""" 60 months""",…,null,null,null,null,null,null


# Step 1.2: Data Sanitization, Target Engineering & Leakage Audit

### Objectives
1. **Schema Sanitization:** Coerce the `id` column to `pl.String` and drop malformed/corrupted rows (e.g., rogue header strings or footer metadata injected during CSV generation).
2. **Deterministic Target Definition:** Isolate historical loan outcomes into binary default indicator (`is_default`).
3. **Data Leakage Audit:** Strip out post-origination collection variables (e.g., `recoveries`, `total_rec_prncp`, `last_pymnt_d`) that contain future information inaccessible at loan underwriting time.

### Target Mapping Rules
- **Default (`1`):** `Charged Off`, `Default`, `Late (31-120 days)`, `Does not meet the credit policy. Status:Charged Off`
- **Non-Default (`0`):** `Fully Paid`, `Does not meet the credit policy. Status:Fully Paid`
- **Exclusions:** `Current`, `In Grace Period`, `Late (16-30 days)` (indeterminate performance window)

In [8]:
# Empirical Discovery: Scan unique loan statuses and their exact counts
loan_status_distribution = (
    lazy_df
    # Ensure ID is coerced to string and rogue rows are dropped first
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    
    # Group and count occurrences across the entire raw dataset
    .group_by("loan_status")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .collect()
)

print("Empirical Unique loan_status Distribution:")
print(loan_status_distribution)

Empirical Unique loan_status Distribution:
shape: (9, 2)
┌─────────────────────────────────┬─────────┐
│ loan_status                     ┆ count   │
│ ---                             ┆ ---     │
│ str                             ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ Fully Paid                      ┆ 1076751 │
│ Current                         ┆ 878317  │
│ Charged Off                     ┆ 268559  │
│ Late (31-120 days)              ┆ 21467   │
│ In Grace Period                 ┆ 8436    │
│ Late (16-30 days)               ┆ 4349    │
│ Does not meet the credit polic… ┆ 1988    │
│ Does not meet the credit polic… ┆ 761     │
│ Default                         ┆ 40      │
└─────────────────────────────────┴─────────┘


In [11]:
#rows with text in the id column
total_raw_rows = lazy_df.select(pl.len()).collect().item()

sanitized_row_count = (
    lazy_df
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    .select(pl.len())
    .collect()
    .item()
)

dropped_rows = total_raw_rows - sanitized_row_count
print(f"Total Raw Rows in CSV: {total_raw_rows:,}")
print(f"Rows After ID Sanitization: {sanitized_row_count:,}")
print(f"Dropped Malformed/Rogue Rows: {dropped_rows:,} ({(dropped_rows / total_raw_rows):.4%})")

Total Raw Rows in CSV: 2,260,701
Rows After ID Sanitization: 2,260,668
Dropped Malformed/Rogue Rows: 33 (0.0015%)


In [12]:
# Discover how loan statuses are distributed temporally across issue years
status_by_year_discovery = (
    lazy_df
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    # Parse year from issue_d (format like Dec-2015)
    .with_columns(pl.col("issue_d").str.to_date("%b-%Y", strict=False).dt.year().alias("issue_year"))
    .group_by(["issue_year", "loan_status"])
    .agg(pl.len().alias("count"))
    .sort(["issue_year", "count"], descending=[False, True])
    .collect()
)

print(status_by_year_discovery)

shape: (60, 3)
┌────────────┬─────────────────────────────────┬────────┐
│ issue_year ┆ loan_status                     ┆ count  │
│ ---        ┆ ---                             ┆ ---    │
│ i32        ┆ str                             ┆ u32    │
╞════════════╪═════════════════════════════════╪════════╡
│ 2007       ┆ Does not meet the credit polic… ┆ 239    │
│ 2007       ┆ Fully Paid                      ┆ 206    │
│ 2007       ┆ Does not meet the credit polic… ┆ 113    │
│ 2007       ┆ Charged Off                     ┆ 45     │
│ 2008       ┆ Fully Paid                      ┆ 1315   │
│ 2008       ┆ Does not meet the credit polic… ┆ 582    │
│ 2008       ┆ Does not meet the credit polic… ┆ 249    │
│ 2008       ┆ Charged Off                     ┆ 247    │
│ …          ┆ …                               ┆ …      │
│ 2018       ┆ Current                         ┆ 427181 │
│ 2018       ┆ Fully Paid                      ┆ 47444  │
│ 2018       ┆ Charged Off                     ┆ 8867   │

# Step 1.3: Target Mapping & Temporal Out-of-Time (OOT) Split

### Objective
1. **Target Engineering:** Map our empirically verified statuses into a binary target (`is_default`).
   - `is_default = 1`: `Charged Off`, `Default`, `Late (31-120 days)`
   - `is_default = 0`: `Fully Paid`, along with legacy credit policy exceptions that resolved successfully.
   - *Exclusions:* Drop indeterminate active states (`Current`, `In Grace Period`, `Late (16-30 days)`).
2. **Temporal Partitioning:** Split the historical data before and after 2018 to create an Out-of-Time (OOT) validation set, preventing future data leakage.

In [13]:
# 1. Define empirical target mappings based on discovery
DEFAULT_STATUSES = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Does not meet the credit policy. Status:Charged Off"
]

PAID_STATUSES = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]
# 2. Execute target mapping and temporal feature extraction on the LazyFrame
modeled_lazy = (
    lazy_df
    # Sanitize ID
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    
    # Filter to only include resolved states
    .filter(pl.col("loan_status").is_in(DEFAULT_STATUSES + PAID_STATUSES))
    
    # Create binary target
    .with_columns(
        pl.when(pl.col("loan_status").is_in(DEFAULT_STATUSES))
        .then(1)
        .otherwise(0)
        .alias("is_default")
    )
    
    # Extract issue year for temporal split
    .with_columns(
        pl.col("issue_d").str.to_date("%b-%Y", strict=False).dt.year().alias("issue_year")
    )
)

# 3. Collect into memory for partitioning
processed_df = modeled_lazy.collect()

# 4. Enforce OOT split (e.g., Development: < 2018, OOT Evaluation: >= 2018)
train_df = processed_df.filter(pl.col("issue_year") < 2018)
oot_df = processed_df.filter(pl.col("issue_year") >= 2018)

print(f"Total Processed Resolved Loans: {processed_df.height:,}")
print(f"In-Time Training Set (< 2018): {train_df.height:,} rows | Default Rate: {train_df['is_default'].mean():.2%}")
print(f"Out-of-Time Evaluation Set (>= 2018): {oot_df.height:,} rows | Default Rate: {oot_df['is_default'].mean():.2%}")

Total Processed Resolved Loans: 1,369,566
In-Time Training Set (< 2018): 1,306,027 rows | Default Rate: 21.04%
Out-of-Time Evaluation Set (>= 2018): 63,539 rows | Default Rate: 25.33%


# Phase 2: Feature Engineering & Missingness Discovery

### Objective
Profile the training set feature space for null rates and structural missingness. We will identify which columns are densely populated versus those that require omission or specialized imputation strategies.

In [16]:
# Calculate null counts and null percentages across all columns in the training set
missingness_df = (
    train_df
    .select([
        (pl.col(col).null_count() / pl.len()).alias(col) 
        for col in train_df.columns
    ])
    .transpose(include_header=True, header_name="feature", column_names=["null_pct"])
    .sort("null_pct", descending=True)
)

print("Top 20 Columns with Highest Missingness:")
print(missingness_df.head(20))

Top 20 Columns with Highest Missingness:
shape: (20, 2)
┌─────────────────────────────────┬──────────┐
│ feature                         ┆ null_pct │
│ ---                             ┆ ---      │
│ str                             ┆ f64      │
╞═════════════════════════════════╪══════════╡
│ member_id                       ┆ 1.0      │
│ sec_app_mths_since_last_major_… ┆ 0.996511 │
│ orig_projected_additional_accr… ┆ 0.996144 │
│ hardship_type                   ┆ 0.994515 │
│ hardship_reason                 ┆ 0.994515 │
│ hardship_status                 ┆ 0.994515 │
│ deferral_term                   ┆ 0.994515 │
│ hardship_amount                 ┆ 0.994515 │
│ …                               ┆ …        │
│ hardship_loan_status            ┆ 0.994515 │
│ hardship_payoff_balance_amount  ┆ 0.994515 │
│ hardship_last_payment_amount    ┆ 0.994515 │
│ sec_app_revol_util              ┆ 0.990647 │
│ revol_bal_joint                 ┆ 0.990492 │
│ sec_app_fico_range_low          ┆ 0.990491 │
│ se

# Step 2.1: Feature Pruning & Leakage Prevention

### Objective
Drop columns exhibiting extreme missingness (>90% null) and strip out post-origination leakage features to ensure our feature store contains only variables known at the exact moment of loan underwriting.

In [18]:
# 1. Identify columns with > 90% missingness in the training set
high_null_cols = (
    missingness_df
    .filter(pl.col("null_pct") > 0.90)
    .get_column("feature")
    .to_list()
)

# 2. Define known post-origination leakage columns to strip
LEAKAGE_COLUMNS = [
    "collection_recovery_fee", "funded_amnt", "funded_amnt_inv", 
    "issue_d", "last_credit_pull_d", "last_fico_band_high", "last_fico_band_low",
    "last_pymnt_amnt", "last_pymnt_d", "next_pymnt_d", "out_prncp",
    "out_prncp_inv", "recoveries", "total_pymnt", "total_pymnt_inv",
    "total_rec_int", "total_rec_late_fee", "total_rec_prncp",
    "policy_code", "url", "id", "emp_title", "title" # Drop text/ID identifiers
]

# Combine columns to drop
columns_to_drop = list(set(high_null_cols + LEAKAGE_COLUMNS))

print(f"Total features originally: {train_df.width}")
print(f"Total columns targeted for removal (high nulls + leakage/identifiers): {len(columns_to_drop)}")

Total features originally: 153
Total columns targeted for removal (high nulls + leakage/identifiers): 61


In [19]:
# 3. Filter feature set for train and oot sets (retaining 'is_default' and 'issue_year')
final_train_df = train_df.drop([c for c in columns_to_drop if c in train_df.columns])
final_oot_df = oot_df.drop([c for c in columns_to_drop if c in oot_df.columns])

print(f"Final feature count for modeling: {final_train_df.width - 2}") # excluding target and year

Final feature count for modeling: 92


# Phase 3: Categorical Encoding & Baseline Model Training

### Objective
1. Identify remaining categorical string columns in our 92-feature set.
2. Encode categorical variables into integer/ordinal formats suitable for tree-based algorithms.
3. Separate feature matrices ($X$) and target vectors ($y$) for both training and OOT evaluation sets.

In [20]:
# 1. Identify remaining string columns in the training set (excluding target and year)
categorical_cols = [
    col for col, dtype in zip(final_train_df.columns, final_train_df.dtypes) 
    if dtype == pl.Utf8 and col not in ["is_default"]
]

print(f"Identified Categorical Columns to Encode: {categorical_cols}")

Identified Categorical Columns to Encode: ['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'loan_status', 'pymnt_plan', 'purpose', 'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status', 'application_type', 'hardship_flag', 'disbursement_method', 'debt_settlement_flag']


In [21]:
# 2. Convert categorical columns to categorical dtype / integer codes in Polars
# Using Polars .cast(pl.Categorical).to_physical() for efficient tree-friendly encoding
encoded_train_df = final_train_df.with_columns([
    pl.col(col).cast(pl.Categorical).to_physical().alias(col)
    for col in categorical_cols
])

encoded_oot_df = final_oot_df.with_columns([
    pl.col(col).cast(pl.Categorical).to_physical().alias(col)
    for col in categorical_cols
])

# 3. Separate Features (X) and Target (y)
feature_cols = [c for c in encoded_train_df.columns if c not in ["is_default", "issue_year"]]

X_train = encoded_train_df.select(feature_cols).to_numpy()
y_train = encoded_train_df.select("is_default").to_numpy().ravel()

X_oot = encoded_oot_df.select(feature_cols).to_numpy()
y_oot = encoded_oot_df.select("is_default").to_numpy().ravel()

print(f"Training Matrix Shape (X_train): {X_train.shape}")
print(f"OOT Evaluation Matrix Shape (X_oot): {X_oot.shape}")

Training Matrix Shape (X_train): (1306027, 92)
OOT Evaluation Matrix Shape (X_oot): (63539, 92)


# Phase 3.1: XGBoost Baseline Model Training & OOT Evaluation

### Objective
1. Ensure `loan_status` and any text variables are strictly stripped from the feature matrix to prevent target leakage.
2. Train an XGBoost classifier on the pre-2018 training dataset (`X_train`, `y_train`).
3. Evaluate predictive power on the Out-of-Time holdout set (`X_oot`, `y_oot`) using **ROC-AUC / Gini** and the **Kolmogorov-Smirnov (KS) statistic**—the gold standards in credit risk scorecard validation.

In [22]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import numpy as np

# 1. Ensure loan_status is explicitly dropped from features if present
strict_feature_cols = [
    c for c in feature_cols 
    if c not in ["loan_status", "is_default", "issue_year"]
]

X_train_final = encoded_train_df.select(strict_feature_cols).to_numpy()
X_oot_final = encoded_oot_df.select(strict_feature_cols).to_numpy()

print(f"Strict Feature Count for Modeling: {len(strict_feature_cols)}")

# 2. Configure and train XGBoost classifier
# Using optimized parameters for tabular credit risk modeling
params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "tree_method": "hist" # Fast histogram-optimized splitting for large datasets
}

dtrain = xgb.DMatrix(X_train_final, label=y_train, feature_names=strict_feature_cols)
doot = xgb.DMatrix(X_oot_final, label=y_oot, feature_names=strict_feature_cols)

# Train with early stopping or fixed round count for baseline
evals = [(dtrain, "train"), (doot, "oot")]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=300,
    evals=evals,
    verbose_eval=50
)

# 3. Predict probabilities on OOT set
oot_preds = model.predict(doot)

# Calculate Gini coefficient from ROC-AUC (Gini = 2 * AUC - 1)
oot_auc = roc_auc_score(y_oot, oot_preds)
oot_gini = (2 * oot_auc) - 1

print(f"\n--- OOT Evaluation Results ---")
print(f"Out-of-Time ROC-AUC: {oot_auc:.4f}")
print(f"Out-of-Time Gini Coefficient: {oot_gini:.4f}")

Strict Feature Count for Modeling: 91
[0]	train-auc:0.94983	oot-auc:0.15196
[50]	train-auc:0.95775	oot-auc:0.30463
[100]	train-auc:0.95941	oot-auc:0.58542
[150]	train-auc:0.96022	oot-auc:0.59933
[200]	train-auc:0.96087	oot-auc:0.61948
[250]	train-auc:0.96139	oot-auc:0.62937
[299]	train-auc:0.96187	oot-auc:0.63946

--- OOT Evaluation Results ---
Out-of-Time ROC-AUC: 0.6395
Out-of-Time Gini Coefficient: 0.2789
